## 2.5 Prevención de fuga de datos (*Data Leakage*)

### 2.5.1 Disponibilidad en el instante de predicción

La entrada original es `close`. Para una vela cuya apertura es s, su cierre solo está disponible una vez finalizada la hora, alrededor de s+1 hora. La predicción se realiza después de recibir ese cierre, no al abrir la vela. Los timestamps del archivo permiten verificar el orden temporal, pero no acreditan la latencia real de publicación o recepción de la API; una implementación operativa deberá registrar esa disponibilidad.

| Campo o construcción | Disponibilidad y función | Decisión |
|---|---|---|
| `close` de la última vela cerrada | Observado antes de predecir. No se usa el cierre de una vela en curso. | Entrada admitida. |
| Rezagos de `close` | Disponibles si pertenecen a velas anteriores ya cerradas. | Solo si se define una ventana común; no implican nuevas variables de origen. |
| Retornos y volatilidad pasada | Disponibles si se calculan exclusivamente hacia atrás con cierres consecutivos. | Derivados exploratorios; no se incorporan automáticamente al modelo. |
| Volatilidad futura de 24 horas | Requiere 24 retornos posteriores y se conoce al finalizar el horizonte. | Etiqueta; excluida de las entradas. |
| `symbol` | Identifica el activo. | Clave de agrupación, no predictor en este diagnóstico. |
| `open_time`, `close_time` | Ordenan las velas y verifican su cierre. | Metadatos de alineación, no predictores. |
| Resto de campos de mercado | No forman parte de la entrada definida para el proyecto. | Excluidos por alcance, sin afirmar que sean fugas por sí mismos. |

### 2.5.2 Objetivo, derivados y posibles proxies

El objetivo se calcula como 100 veces la desviación estándar de los 24 retornos logarítmicos futuros, centrados en su media y con divisor 24. A partir del ancla s utiliza los retornos s+1 a s+24. El desplazamiento hacia el futuro se usa únicamente para construir la etiqueta; no se permite en las entradas.

Compartir el cierre de frontera entre el último dato observado y el primer retorno futuro no constituye fuga: ese precio ya se conoce al predecir. En cambio, usar cierres futuros, ventanas centradas, rellenos hacia atrás desde el futuro, la propia etiqueta o transformaciones de ella como predictores introduciría información no disponible. No se usan esas construcciones en el diagnóstico de esta sección. La correlación o información mutua por sí solas no demuestran disponibilidad temporal ni ausencia de fuga.

### 2.5.3 Diagnóstico predictivo de close

AUC no aplica al objetivo continuo de regresión. Se ajusta una regresión lineal diagnóstica con intercepto y una sola entrada, `close`, sin búsqueda de hiperparámetros. Se reserva el último 20 % de los timestamps observados de DEVELOPMENT para validación interna, con frontera común en **2024-07-09 20:00 UTC** (hora de apertura de la vela ancla). No es el TEST final ni una comparación definitiva de modelos.

El escalado de `close` utiliza únicamente el entrenamiento. Se excluyen 24 etiquetas por activo en la frontera: la última vela requerida por cada etiqueta de entrenamiento debe preceder al primer ancla de validación. Los huecos y cierres irregulares invalidan las ventanas correspondientes; no se imputan. Tampoco se construyen etiquetas que excedan DEVELOPMENT. RMSE y MAE se expresan en puntos porcentuales de volatilidad; R² es adimensional. La referencia constante utiliza solo la media de las etiquetas de entrenamiento.

| Activo | Entrenamiento | Validación | RMSE close | MAE close | R² close | RMSE media de entrenamiento |
|---|---:|---:|---:|---:|---:|---:|
| BNBUSDT | 33972 | 8543 | 0.3551 | 0.2962 | -0.4413 | 0.3683 |
| BTCUSDT | 33972 | 8543 | 0.3362 | 0.2891 | -0.9333 | 0.2594 |
| ETHUSDT | 33972 | 8543 | 0.3512 | 0.2561 | -0.0173 | 0.3510 |
| SOLUSDT | 33997 | 8543 | 0.4685 | 0.3724 | -0.2479 | 0.5533 |
| XRPUSDT | 33997 | 8543 | 0.9824 | 0.8353 | -1.4772 | 0.6242 |

Los R² son negativos en los cinco activos: esta relación lineal con un único cierre no explica bien la volatilidad futura en el bloque evaluado. No aparece un desempeño casi perfecto que active esa señal de alarma. Sin embargo, un desempeño bajo no prueba ausencia de fuga ni descarta que ventanas históricas o relaciones no lineales tengan utilidad. Un R² negativo compara con la media del bloque evaluado; la referencia de la última columna es distinta porque utiliza la media de entrenamiento.

### 2.5.4 Duplicados, entidades y fronteras temporales

La comprobación actual en DEVELOPMENT encuentra **0 filas duplicadas exactas** y **0 claves (`symbol`, `open_time`) duplicadas**. Los registros existentes de la partición sitúan su última apertura en **2025-07-01 18:00 UTC** y la primera de TEST en **2025-07-01 19:00 UTC**, con el mismo corte para los cinco activos y sin mezcla aleatoria.

Según esos registros, los intervalos no se solapan; por tanto, no comparten claves temporales ni filas completas idénticas que incluyan esas fechas. Esta conclusión utiliza la auditoría guardada de la partición, no una nueva lectura de TEST. No se certifica aquí la ausencia de vectores de precios iguales o casi duplicados entre archivos ignorando la fecha. Un cierre repetido, por sí solo, no identifica un registro duplicado.

BTC, ETH, BNB, XRP y SOL aparecen en ambas particiones. Esta repetición de entidades es coherente con pronosticar el futuro de los mismos activos; no evalúa generalización a criptomonedas desconocidas. Los activos de una misma hora deberán permanecer en el mismo bloque en la validación temporal.

La separación de filas por fecha no basta para separar etiquetas futuras. En cada fold se deben purgar las etiquetas cuyo periodo objetivo invada la validación. El diagnóstico ejecutado comprueba esa condición por fechas reales. Los folds definitivos y sus preprocesadores deberán superar la misma verificación cuando se construyan. Utilizar historia anterior al corte como contexto de una predicción posterior es admisible si ya estaba disponible; entrenar con sus resultados futuros no lo es.

### 2.5.5 Transformaciones y alcance de la evidencia

Escaladores, imputaciones, PCA, selección de variables, umbrales e hiperparámetros deberán ajustarse exclusivamente dentro del entrenamiento de cada fold. El EDA previo sobre DEVELOPMENT completo no equivale a una validación fuera de muestra: sus estadísticas y transformaciones auxiliares no deben reutilizarse como preprocesadores ya ajustados. Si sus conclusiones guían decisiones, las métricas internas tienen ese contexto exploratorio y TEST debe conservarse para la evaluación final.

En este diagnóstico se verificaron el orden temporal, la purga, la disponibilidad de las etiquetas dentro de DEVELOPMENT y el ajuste del escalado solo en entrenamiento. No se audita todavía un pipeline final entrenado ni la latencia de una implementación en producción. Las cifras antiguas de 2.1 no se reutilizan: el objetivo se recalcula con la definición común de desviación estándar.

### 2.5.6 Interpretación y decisiones

Se conserva `close` de velas cerradas como entrada original. Se excluyen de la entrada el objetivo, sus transformaciones y cualquier observación futura. Identificadores y fechas se mantienen como claves; los demás campos quedan fuera del alcance acordado. Las ventanas de rezagos y los derivados históricos quedan sujetos a comprobar su disponibilidad y a una definición común antes de comparar modelos.

Las comprobaciones realizadas no detectan infracciones de las reglas temporales implementadas en el diagnóstico. Esta evidencia no constituye una garantía general de ausencia de fuga: siguen siendo necesarias la auditoría de los folds y pipelines definitivos y la verificación operativa de disponibilidad. No se utiliza TEST para elegir variables, ventanas o configuraciones.

### 2.5.7 Reproducibilidad

El procedimiento está en `src/13_leakage_audit.py`. Las métricas se guardan en `outputs/tables/leakage_close_diagnostic.csv` y la trazabilidad en `outputs/tables/leakage_audit_metadata.json`. Se verifica que el SHA-256 de DEVELOPMENT no cambie durante la ejecución. Los datos no se modifican y el archivo TEST no se lee; sus fronteras se consultan exclusivamente en los registros existentes de la partición.


In [1]:
"""Auditoría temporal y diagnóstico univariado; solo DEVELOPMENT."""
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/splits/development_80.csv').is_file())
def main():
    source=ROOT/'data/splits/development_80.csv'
    sha=hashlib.sha256(source.read_bytes()).hexdigest()
    df=pd.read_csv(source)
    df['open_time']=pd.to_datetime(df.open_time,utc=True)
    df['close_time']=pd.to_datetime(df.close_time,utc=True,format='ISO8601')
    times=pd.DatetimeIndex(sorted(df.open_time.unique()))
    boundary=times[int(len(times)*.8)]
    grid=pd.date_range(times.min(),times.max(),freq='h')
    rows=[]
    for symbol,g in df.groupby('symbol'):
        g=g.set_index('open_time').reindex(grid)
        c=g.close.where(g.close_time.eq(grid+pd.Timedelta(hours=1)-pd.Timedelta(milliseconds=1)))
        r=np.log(c).diff()
        y=100*r.rolling(24,min_periods=24).std(ddof=0).shift(-24)
        sample=pd.DataFrame({'close':c,'target':y}).dropna()
        # Última vela objetivo debe ser anterior al primer ancla de validación.
        train=sample.loc[sample.index+pd.Timedelta(hours=24)<boundary]
        val=sample.loc[sample.index>=boundary]
        purged=((sample.index<boundary)&(sample.index+pd.Timedelta(hours=24)>=boundary)).sum()
        assert train.index.max()+pd.Timedelta(hours=24)<val.index.min()
        assert sample.index.max()+pd.Timedelta(hours=24)<=times.max()
        mean=train.close.mean();scale=train.close.std(ddof=0)
        x=np.column_stack([np.ones(len(train)),(train.close-mean)/scale])
        beta=np.linalg.lstsq(x,train.target,rcond=None)[0]
        pred=beta[0]+beta[1]*(val.close.to_numpy()-mean)/scale
        real=val.target.to_numpy();base=np.full(len(real),train.target.mean())
        def rmse(p): return float(np.sqrt(np.mean((real-p)**2)))
        r2=1-np.sum((real-pred)**2)/np.sum((real-real.mean())**2)
        rows.append(dict(symbol=symbol,n_train=len(train),n_validation=len(val),purged=int(purged),rmse_close=rmse(pred),mae_close=float(np.mean(abs(real-pred))),r2_close=float(r2),rmse_train_mean=rmse(base)))
    summary=pd.DataFrame(rows)
    split=pd.read_csv(ROOT/'outputs/tables/chronological_split_audit.csv').set_index('split')
    assert pd.Timestamp(split.loc['DEVELOPMENT','end_time'])<pd.Timestamp(split.loc['TEST','start_time'])
    assert times.max()==pd.Timestamp(split.loc['DEVELOPMENT','end_time'])
    assert sha==hashlib.sha256(source.read_bytes()).hexdigest()
    meta={'development_sha256':sha,'validation_anchor_start':str(boundary),'development_end':str(times.max()),'duplicate_rows_development':int(df.duplicated().sum()),'duplicate_keys_development':int(df.duplicated(['symbol','open_time']).sum()),'test_read':False,'cross_partition_evidence':'Existing split metadata only; no fresh TEST scan','diagnostic':'OLS close only, intercept, chronological 80/20 within DEVELOPMENT, purge by target end, training-only scale','target':'100 * future rolling std of 24 hourly log returns, ddof=0'}
    out=ROOT/'outputs/tables'
    summary.to_csv(out/'leakage_close_diagnostic.csv',index=False)
    (out/'leakage_audit_metadata.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
    print(summary.to_string(index=False));print(json.dumps(meta,indent=2,ensure_ascii=False))
    return summary,meta
if __name__=='__main__':
    summary, metadata = main()



 symbol  n_train  n_validation  purged  rmse_close  mae_close  r2_close  rmse_train_mean
BNBUSDT    33972          8543      24    0.355111   0.296194 -0.441250         0.368254
BTCUSDT    33972          8543      24    0.336171   0.289144 -0.933254         0.259376
ETHUSDT    33972          8543      24    0.351193   0.256112 -0.017318         0.350991
SOLUSDT    33997          8543      24    0.468450   0.372359 -0.247928         0.553315
XRPUSDT    33997          8543      24    0.982425   0.835342 -1.477161         0.624200
{
  "development_sha256": "5fbbc54a12e976551b35652b72c989be8d1a120ef41cd7c6a4b50dcaf5fa87a2",
  "validation_anchor_start": "2024-07-09 20:00:00+00:00",
  "development_end": "2025-07-01 18:00:00+00:00",
  "duplicate_rows_development": 0,
  "duplicate_keys_development": 0,
  "test_read": false,
  "cross_partition_evidence": "Existing split metadata only; no fresh TEST scan",
  "diagnostic": "OLS close only, intercept, chronological 80/20 within DEVELOPMENT, purge 